# Year-by-year housing and retirement report

Edit the inputs below, then run all cells. Make only the required monthly mortgage payments and invest all surplus. At retirement, pay off whatever mortgage principal remains from investments.

Retirement is reached when investments left after that payoff cover mortgage-free annual expenses at your chosen withdrawal rate. Property taxes, insurance and maintenance continue, and expenses increase with inflation.


In [1]:
from utils import Buyer, Economy, House, to_today_dollars

# Display only: True = today's purchasing power; False = future nominal dollars.
show_today_dollars = True

buyer = Buyer(
    assets=300_000,  # Available money BEFORE down payment and closing costs; exclude home equity.
    yearlyIncomeAfterTax=200_000,  # Annual take-home income before living/housing expenses.
    yearlyExpensesWithoutHouse=50_000,  # Annual living costs; exclude rent that stops after buying.
    withdrawalRate=0.03,  # Annual withdrawals / invested balance: 3% means a 33.3x spending target.
)

economy = Economy(
    investmentReturn=0.07,  # Annual nominal investment growth, net of fees and annual tax drag.
    interest_rate=0.075,  # Fixed annual mortgage interest rate, NOT APR.
    inflation_rate=0.035,  # Annual increase in living and ongoing housing expenses.
    income_growth_rate=0.035,  # Annual growth in take-home income; set 0 for flat nominal wages.
    realEstateTax=0.01,  # Starting annual property tax / house price.
    insuranceRatio=0.0015,  # Starting annual homeowners insurance / house price.
)

house = House(
    price=1_000_000,
    economy=economy,
    down_payment_ratio=0.2,  # Fraction of price paid out of the assets above.
    loan_years=30,
    maintenance_cost_ratio=0.01,  # Starting annual maintenance budget / house price.
    closing_cost_ratio=0.04,  # 4% of price, also paid out of starting assets.
)


## Annual report

- **Liquid assets:** investments plus cash, excluding the house.
- **Liquid net worth:** liquid assets minus remaining mortgage principal, before any extra withdrawal taxes.
- **Total net worth:** includes the house, held at purchase price before conversion into the selected dollar basis. No appreciation is assumed.
- **Total expenses:** actual living costs, ongoing home costs, and required mortgage payments during that period. Mortgage principal is included as a cash outflow, although paying principal builds home equity.
- **Principal remaining:** mortgage capital still owed at the end of that period.

Balances are measured at each period's end. The final working period may be less than a full year. Down payment and closing costs are shown separately; the purchase row has no elapsed-period expenses. The final payoff is also shown separately, followed by the balances after it. Set `show_today_dollars = True` to show purchasing power in today's dollars, or `False` for nominal amounts at each future date. The active mode is printed above the table. Balances are adjusted for inflation at their reported date; expense payments are adjusted month by month before being added. The retirement date is unchanged by this display setting. Because the house value is fixed nominally, its inflation-adjusted value declines over time.


In [2]:
from IPython.display import Markdown, display

# One plan: regular mortgage payments, all surplus invested, final payoff at retirement.
projection = buyer.project(house)
annual_rows = buyer.annual_report(house, projection, today_dollars=show_today_dollars)

def report_amount(value, month):
    return to_today_dollars(value, month, economy.inflation_rate) if show_today_dollars else value

basis = "TODAY'S DOLLARS (inflation-adjusted purchasing power)" if show_today_dollars else 'NOMINAL DOLLARS (at each future date)'
print(f'Report amounts: {basis}')

print(f'Down payment: ${house.down_payment:,.0f}')
print(f'Closing costs: ${house.closing_costs:,.0f}')
print(f'Initial monthly mortgage principal + interest: ${house.monthly_payment:,.2f}')

if not projection.affordability.affordable:
    print('Purchase requirements not met:')
    for reason in projection.affordability.reasons:
        print('-', reason)
else:
    table = [
        '| Period end | Liquid assets | Liquid net worth | Total net worth | Total expenses in period | Principal remaining |',
        '| :--- | ---: | ---: | ---: | ---: | ---: |',
    ]
    for row in annual_rows:
        years, months = divmod(row.month, 12)
        if row.month == 0:
            period = 'At purchase'
        elif months == 0:
            period = f'Year {years}'
        else:
            period = f'Year {years + 1} ({row.months_in_period} months only)'
        table.append(f'| {period} | ${row.liquid_assets:,.0f}'
                     f' | ${row.liquid_net_worth:,.0f} | ${row.total_net_worth:,.0f}'
                     f' | ${row.total_expenses:,.0f} | ${row.mortgage_balance:,.0f} |')

    if projection.months_to_retirement is not None:
        final = projection.history[-1]
        retirement_cash = buyer.emergency_fund_months * final.mortgage_free_annual_spending / 12
        liquid_after_payoff = final.investments_after_payoff + retirement_cash
        table.append(f'| After retirement payoff | ${report_amount(liquid_after_payoff, final.month):,.0f}'
                     f' | ${report_amount(liquid_after_payoff, final.month):,.0f} | ${report_amount(liquid_after_payoff + house.price, final.month):,.0f}'
                     ' | Separate payoff below | $0 |')
    display(Markdown('\n'.join(table)))

    if projection.months_to_retirement is not None:
        years, months = divmod(projection.months_to_retirement, 12)
        print(f'Retirement target reached after {years} years, {months} months.')
        print(f'Mortgage principal paid at retirement: ${report_amount(final.mortgage_balance, final.month):,.0f}')
        if final.gross_payoff_withdrawal != final.mortgage_balance:
            print(f'Investment withdrawal for payoff including modeled tax: ${report_amount(final.gross_payoff_withdrawal, final.month):,.0f}')
        print(f'Mortgage-free annual expenses at retirement: ${report_amount(final.mortgage_free_annual_spending, final.month):,.0f}')
        print(f'Required invested retirement balance: ${report_amount(final.spending_target, final.month):,.0f}')
        print(f'Investments remaining after payoff: ${report_amount(final.investments_after_payoff, final.month):,.0f}')
    else:
        print(f'Retirement target not reached: {projection.status}')


Report amounts: TODAY'S DOLLARS (inflation-adjusted purchasing power)
Down payment: $200,000
Closing costs: $40,000
Initial monthly mortgage principal + interest: $5,593.72


| Period end | Liquid assets | Liquid net worth | Total net worth | Total expenses in period | Principal remaining |
| :--- | ---: | ---: | ---: | ---: | ---: |
| At purchase | $60,000 | $-740,000 | $260,000 | $0 | $800,000 |
| Year 1 | $125,225 | $-640,596 | $325,587 | $137,578 | $765,822 |
| Year 2 | $194,919 | $-537,587 | $395,924 | $135,343 | $732,505 |
| Year 3 | $269,155 | $-430,855 | $471,087 | $133,184 | $700,010 |
| Year 4 | $348,014 | $-320,282 | $551,160 | $131,098 | $668,296 |
| Year 5 | $431,581 | $-205,742 | $636,231 | $129,083 | $637,323 |
| Year 6 | $519,945 | $-87,107 | $726,393 | $127,136 | $607,052 |
| Year 7 | $613,202 | $35,757 | $821,747 | $125,254 | $577,446 |
| Year 8 | $711,454 | $162,987 | $922,399 | $123,437 | $548,467 |
| Year 9 | $814,807 | $294,728 | $1,028,459 | $121,680 | $520,078 |
| Year 10 | $923,373 | $431,128 | $1,140,047 | $119,983 | $492,245 |
| Year 11 | $1,037,271 | $572,340 | $1,257,286 | $118,344 | $464,930 |
| Year 12 | $1,156,624 | $718,524 | $1,380,307 | $116,760 | $438,100 |
| Year 13 | $1,281,563 | $869,843 | $1,509,248 | $115,229 | $411,719 |
| Year 14 | $1,412,224 | $1,026,470 | $1,644,252 | $113,750 | $385,755 |
| Year 15 | $1,548,751 | $1,188,579 | $1,785,470 | $112,322 | $360,172 |
| Year 16 | $1,691,293 | $1,356,355 | $1,933,061 | $110,941 | $334,938 |
| Year 17 | $1,840,005 | $1,529,986 | $2,087,190 | $109,607 | $310,019 |
| Year 18 | $1,995,051 | $1,709,668 | $2,248,029 | $108,319 | $285,383 |
| Year 19 | $2,156,601 | $1,895,603 | $2,415,759 | $107,074 | $260,998 |
| Year 20 | $2,324,832 | $2,088,002 | $2,590,568 | $105,871 | $236,830 |
| Year 21 | $2,499,929 | $2,287,082 | $2,772,653 | $104,708 | $212,847 |
| Year 22 (6 months only) | $2,590,112 | $2,389,197 | $2,866,487 | $51,931 | $200,915 |
| After retirement payoff | $2,389,197 | $2,389,197 | $2,866,487 | Separate payoff below | $0 |

Retirement target reached after 21 years, 6 months.
Mortgage principal paid at retirement: $200,915
Mortgage-free annual expenses at retirement: $71,500
Required invested retirement balance: $2,383,333
Investments remaining after payoff: $2,389,197


The report stops at the retirement date, or at the search limit if the target is not reached. The annual expense column sums actual cash outflows in each elapsed period; the mortgage-free annual expense shown below the table is the forward spending budget at retirement.

Inflation increases expenses; investmentReturn is nominal growth before inflation adjustment. The basic inputs assume no extra withdrawal taxes or emergency reserve, and use the same nonhousing budget while working and retired. The return and withdrawal rate remain assumptions, not guarantees.
